# Project 3 — Chronic Disease Cost Modelling
## Notebook 3: SQL Views

**Author:** Amr Thabet  
**Dataset:** chronic_disease_cohort.csv — 3,000 members, 2023–2024  

### Objective
Six analyst-grade SQL queries that answer real GCC health insurance business questions.
Each view is written in SQLite via Python and mirrors the kind of analysis 
a UAE payer or RCM team would run on their claims data warehouse.

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(':memory:')
df = pd.read_csv('../data/chronic_disease_cohort.csv')
df.to_sql('chronic_disease_cohort', conn, index=False, if_exists='replace')

print(f"Loaded {len(df):,} members into SQLite")
print(f"Columns: {df.columns.tolist()}")

Loaded 3,000 members into SQLite
Columns: ['member_id', 'year', 'emirate', 'nationality', 'gender', 'age', 'plan_tier', 'payer', 'condition', 'bmi', 'hba1c', 'systolic_bp', 'smoker', 'comorbidities', 'medication_compliance', 'er_visits', 'inpatient_days', 'outpatient_visits', 'specialist_visits', 'total_cost_aed', 'pharmacy_cost_aed', 'procedure_cost_aed', 'consultation_cost_aed', 'risk_tier', 'high_cost_flag']


## View 1 — Cost summary by chronic condition

**Business question:** Which conditions are driving the most cost and which cohorts should we prioritise for intervention?

In [3]:
view1 = pd.read_sql("""
    SELECT
        condition,
        COUNT(member_id)                                        AS total_members,
        ROUND(AVG(total_cost_aed), 0)                          AS avg_cost_aed,
        ROUND(SUM(total_cost_aed), 0)                          AS total_cohort_cost_aed,
        ROUND(AVG(pharmacy_cost_aed), 0)                       AS avg_pharmacy_cost,
        ROUND(AVG(procedure_cost_aed), 0)                      AS avg_procedure_cost,
        ROUND(AVG(er_visits), 1)                               AS avg_er_visits,
        ROUND(AVG(inpatient_days), 1)                          AS avg_inpatient_days,
        ROUND(100.0 * SUM(high_cost_flag) / COUNT(*), 1)       AS high_cost_pct
    FROM chronic_disease_cohort
    GROUP BY condition
    ORDER BY avg_cost_aed DESC
""", conn)

view1.to_csv('../outputs/view1_cost_by_condition.csv', index=False)
view1

,condition,total_members,avg_cost_aed,total_cohort_cost_aed,avg_pharmacy_cost,avg_procedure_cost,avg_er_visits,avg_inpatient_days,high_cost_pct
0,Diabetes + Cardiac,334,72923.0,24356236.0,21573.0,19567.0,1.5,2.1,77.2
1,Cardiac,456,59139.0,26967542.0,17851.0,15409.0,1.6,2.2,48.7
2,Diabetes + Hypertension,502,46719.0,23452992.0,14118.0,12381.0,1.6,2.1,19.1
3,Diabetes,876,32114.0,28131736.0,9590.0,8534.0,1.6,2.1,2.7
4,Hypertension,832,21501.0,17889043.0,6389.0,5704.0,1.6,2.1,0.0


## View 2 — High-cost member profile

**Business question:** Who are our top 20% cost members and what clinical risk factors do they share?

In [4]:
view2 = pd.read_sql("""
    SELECT
        risk_tier,
        high_cost_flag,
        COUNT(member_id)                                        AS members,
        ROUND(AVG(total_cost_aed), 0)                          AS avg_cost_aed,
        ROUND(AVG(age), 1)                                     AS avg_age,
        ROUND(AVG(bmi), 1)                                     AS avg_bmi,
        ROUND(AVG(comorbidities), 1)                           AS avg_comorbidities,
        ROUND(AVG(er_visits), 1)                               AS avg_er_visits,
        ROUND(AVG(inpatient_days), 1)                          AS avg_inpatient_days,
        ROUND(100.0 * SUM(smoker) / COUNT(*), 1)               AS smoker_pct
    FROM chronic_disease_cohort
    GROUP BY risk_tier, high_cost_flag
    ORDER BY avg_cost_aed DESC
""", conn)

view2.to_csv('../outputs/view2_highcost_profile.csv', index=False)
view2

,risk_tier,high_cost_flag,members,avg_cost_aed,avg_age,avg_bmi,avg_comorbidities,avg_er_visits,avg_inpatient_days,smoker_pct
0,High,1,600,75377.0,50.8,31.1,1.7,1.8,2.5,39.3
1,High,0,420,50624.0,48.0,29.5,1.5,1.8,2.2,27.9
2,Medium,0,990,34818.0,46.1,29.9,1.4,1.6,2.0,29.3
3,Low,0,990,20039.0,42.6,28.5,1.2,1.3,1.9,20.7


## View 3 — Payer benchmarking

**Business question:** Which payers carry above-average risk cohorts and how do they compare to the portfolio average?

This view feeds directly into contract negotiations with UAE payers (DHA/HAAD regulated environment).

In [5]:
view3 = pd.read_sql("""
    WITH portfolio_avg AS (
        SELECT ROUND(AVG(total_cost_aed), 0) AS portfolio_avg_cost
        FROM chronic_disease_cohort
    )
    SELECT
        c.payer,
        COUNT(c.member_id)                                      AS members,
        ROUND(AVG(c.total_cost_aed), 0)                        AS avg_cost_aed,
        p.portfolio_avg_cost,
        ROUND(AVG(c.total_cost_aed) - p.portfolio_avg_cost, 0) AS variance_vs_portfolio,
        ROUND(100.0 * (AVG(c.total_cost_aed) - p.portfolio_avg_cost)
              / p.portfolio_avg_cost, 1)                        AS variance_pct,
        ROUND(100.0 * SUM(c.high_cost_flag) / COUNT(*), 1)     AS high_cost_member_pct,
        ROUND(AVG(c.er_visits), 2)                             AS avg_er_visits
    FROM chronic_disease_cohort c
    CROSS JOIN portfolio_avg p
    GROUP BY c.payer, p.portfolio_avg_cost
    ORDER BY avg_cost_aed DESC
""", conn)

view3.to_csv('../outputs/view3_payer_benchmarking.csv', index=False)
view3

,payer,members,avg_cost_aed,portfolio_avg_cost,variance_vs_portfolio,variance_pct,high_cost_member_pct,avg_er_visits
0,Sukoon Insurance,182,43999.0,40266.0,3733.0,9.3,24.7,1.58
1,MetLife,381,41433.0,40266.0,1167.0,2.9,22.6,1.61
2,ADNIC,365,41197.0,40266.0,931.0,2.3,19.2,1.61
3,AXA Gulf,527,41015.0,40266.0,749.0,1.9,19.4,1.62
4,Daman (HAAD),774,39432.0,40266.0,-834.0,-2.1,19.3,1.53
5,BUPA Global,493,39084.0,40266.0,-1182.0,-2.9,19.7,1.45
6,Cigna,278,37996.0,40266.0,-2270.0,-5.6,18.3,1.68


## View 4 — Medication compliance impact

**Business question:** What is the cost difference between compliant and non-compliant members?

This quantifies the ROI of a pharmacy adherence programme in AED per member per year.

In [6]:
view4 = pd.read_sql("""
    SELECT
        medication_compliance,
        condition,
        COUNT(member_id)                                        AS members,
        ROUND(AVG(total_cost_aed), 0)                          AS avg_cost_aed,
        ROUND(AVG(er_visits), 2)                               AS avg_er_visits,
        ROUND(AVG(inpatient_days), 2)                          AS avg_inpatient_days
    FROM chronic_disease_cohort
    GROUP BY medication_compliance, condition
    ORDER BY condition, avg_cost_aed DESC
""", conn)

view4.to_csv('../outputs/view4_compliance_impact.csv', index=False)
view4

,medication_compliance,condition,members,avg_cost_aed,avg_er_visits,avg_inpatient_days
0,Low,Cardiac,118,59973.0,1.67,2.23
1,Medium,Cardiac,181,59828.0,1.57,2.30
2,High,Cardiac,157,57719.0,1.56,2.17
3,High,Diabetes,324,32506.0,1.56,2.20
4,Low,Diabetes,208,32130.0,1.61,2.00
5,Medium,Diabetes,344,31735.0,1.53,1.95
6,Low,Diabetes + Cardiac,72,76785.0,1.72,2.24
7,High,Diabetes + Cardiac,116,76533.0,1.43,2.20
8,Medium,Diabetes + Cardiac,146,68150.0,1.48,2.01
9,Medium,Diabetes + Hypertension,201,47879.0,1.53,2.23


## View 5 — Risk tier summary by emirate

**Business question:** Where geographically is our high-risk population concentrated?

Which emirates need the most targeted care management resources?

In [7]:
view5 = pd.read_sql("""
    SELECT
        emirate,
        risk_tier,
        COUNT(member_id)                                        AS members,
        ROUND(AVG(total_cost_aed), 0)                          AS avg_cost_aed,
        ROUND(SUM(total_cost_aed), 0)                          AS total_cost_aed,
        ROUND(AVG(comorbidities), 1)                           AS avg_comorbidities,
        ROUND(100.0 * SUM(high_cost_flag) / COUNT(*), 1)       AS high_cost_pct
    FROM chronic_disease_cohort
    GROUP BY emirate, risk_tier
    ORDER BY emirate, avg_cost_aed DESC
""", conn)

view5.to_csv('../outputs/view5_risk_by_emirate.csv', index=False)
view5

,emirate,risk_tier,members,avg_cost_aed,total_cost_aed,avg_comorbidities,high_cost_pct
0,Abu Dhabi,High,344,65712.0,22604806.0,1.5,57.8
1,Abu Dhabi,Medium,377,34573.0,13034023.0,1.4,0.0
2,Abu Dhabi,Low,329,20287.0,6674426.0,1.1,0.0
3,Ajman,High,81,61129.0,4951447.0,1.5,56.8
4,Ajman,Medium,78,35011.0,2730889.0,1.5,0.0
5,Ajman,Low,71,20200.0,1434223.0,1.1,0.0
6,Dubai,High,382,64698.0,24714453.0,1.6,59.4
7,Dubai,Medium,352,35160.0,12376392.0,1.4,0.0
8,Dubai,Low,392,20101.0,7879727.0,1.2,0.0
9,Other,High,59,65413.0,3859372.0,1.5,54.2


## View 6 — Cost concentration (the 80/20 rule)

**Business question:** Do 20% of members drive 80% of total spend?

This is the key CFO metric in UAE health insurance — understanding spend concentration
directly shapes intervention prioritisation and budget allocation.

In [8]:
view6 = pd.read_sql("""
    SELECT
        CASE WHEN high_cost_flag = 1 THEN 'Top 20% (high cost)'
             ELSE 'Bottom 80%' END                              AS member_segment,
        COUNT(member_id)                                        AS members,
        ROUND(100.0 * COUNT(member_id) /
              SUM(COUNT(member_id)) OVER (), 1)                AS pct_of_members,
        ROUND(SUM(total_cost_aed), 0)                          AS total_cost_aed,
        ROUND(100.0 * SUM(total_cost_aed) /
              SUM(SUM(total_cost_aed)) OVER (), 1)             AS pct_of_total_spend,
        ROUND(AVG(total_cost_aed), 0)                          AS avg_cost_aed,
        ROUND(AVG(er_visits), 2)                               AS avg_er_visits,
        ROUND(AVG(inpatient_days), 2)                          AS avg_inpatient_days
    FROM chronic_disease_cohort
    GROUP BY high_cost_flag
    ORDER BY high_cost_flag DESC
""", conn)

view6.to_csv('../outputs/view6_cost_concentration.csv', index=False)
view6

,member_segment,members,pct_of_members,total_cost_aed,pct_of_total_spend,avg_cost_aed,avg_er_visits,avg_inpatient_days
0,Top 20% (high cost),600,20.0,45226274.0,37.4,75377.0,1.82,2.51
1,Bottom 80%,2400,80.0,75571275.0,62.6,31488.0,1.51,2.02
